## SCOR KPI Baseline and Performance Scorecard

### Purpose

This notebook creates a baseline performance scorecard aligned with the SCOR (Supply Chain Operations Reference) model. Using the cleaned silver dataset, it quantifies current-state supply chain performance across planning, execution, reliability, and financial outcomes.

The baseline serves three important roles:

1. Provides an objective reference point for variance analysis.
2. Anchors downstream forecasting, risk modeling, and scenario analysis.
3. Enables consistent KPI definitions for Power BI dashboards and decision-maker reporting.

### SCOR-Aligned KPI Framework

#### Plan

Focuses on demand behavior and predictability, informing capacity planning and forecasting strategy:

- Demand volatility.
- Forecastability proxies (e.g., coefficient of variation).

#### Source/Make

Evaluates internal business execution velocity and fulfillment constraints using operational proxies:

- Order-to-Ship Responsiveness.
- Internal Cycle Time Variability.

#### Deliver

Measures service reliability and execution performance:

- On-time delivery rate.
- Late delivery risk rate.
- Lead time distribution and variability.

#### Return

Assesses the financial and operational impact of returns and service failures:

- Return rate.
- Margin erosion associated with returns and delivery issues.

#### Measure

Provides an enterprise-level financial view of performance:

- Total revenue, total profit, and margin.
- Estimated profit leakage attributable to late deliveries and returns.

### Strategic Rationale: Establishing the Performance Baseline

Grounding advanced analytics in a defined SCOR KPI baseline ensures that all subsequent models and dashboards are interpretable, comparable, and directly tied to operational decisions. This reference point allows future improvements to be measured against current performance - converting raw supply chain data into an assessment that tracks both analytical rigor and business outcomes.

## Load Silver Dataset

In [1]:
import pandas as pd
import numpy as np

# Set display options for dataframe
pd.set_option("display.max_columns", 60)

df = pd.read_parquet("/kaggle/input/orders-silver/orders_silver.parquet")
df.head()

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_email,customer_fname,customer_id,customer_lname,customer_password,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,market,order_city,order_country,order_customer_id,order_date,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_state,order_status,product_card_id,product_category_id,product_image,product_name,product_price,product_status,shipping_date,shipping_mode,actual_delivery_date,shipping_delay,processing_days,profit_margin
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,2018-01-31 22:56:00,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class,2018-02-06 22:56:00,-1,3,0.290014
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,2018-01-13 12:27:00,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class,2018-01-23 12:27:00,1,5,-0.800006
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,2018-01-13 12:06:00,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class,2018-01-21 12:06:00,0,4,-0.800013
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,XXXXXXXXX,Tana,19490,Tate,XXXXXXXXX,Home Office,CA,3200 Amber Bend,90027.0,2,Fitness,34.125946,-118.291016,Pacific Asia,Townsville,Australia,19490,2018-01-13 11:45:00,75937,1360,22.940001,0.07,179252,327.75,0.08,1,327.75,304.809998,22.860001,Oceania,Queensland,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class,2018-01-19 11:45:00,-1,3,0.074998
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Orli,19489,Hendricks,XXXXXXXXX,Corporate,PR,8671 Iron Anchor Corners,725.0,2,Fitness,18.253769,-66.037048,Pacific Asia,Townsville,Australia,19489,2018-01-13 11:24:00,75936,1360,29.500000,0.09,179251,327.75,0.45,1,327.75,298.250000,134.210007,Oceania,Queensland,PENDING_PAYMENT,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class,2018-01-17 11:24:00,-2,2,0.449992


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159763 entries, 0 to 159762
Data columns (total 55 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   type                         159763 non-null  object        
 1   days_for_shipping_real       159763 non-null  int64         
 2   days_for_shipment_scheduled  159763 non-null  int64         
 3   benefit_per_order            159763 non-null  float64       
 4   sales_per_customer           159763 non-null  float64       
 5   delivery_status              159763 non-null  object        
 6   late_delivery_risk           159763 non-null  int64         
 7   category_id                  159763 non-null  int64         
 8   category_name                159763 non-null  object        
 9   customer_city                159763 non-null  object        
 10  customer_country             159763 non-null  object        
 11  customer_email            

## SCOR Performance Framework: KPI Definition and Strategic Mapping

This section computes a SCOR-aligned operational baseline scorecard from the curated Silver dataset. The objective is to translate transactional supply chain records into a compact set of metrics that reflect planning stability, internal business execution efficiency, delivery reliability, reverse-logistics exposure, and financial performance.

A key design principle is that KPIs are not computed as isolated metrics - they are designed to be operationally interpretable and decision-oriented, so that subsequent forecasting and risk modeling can be evaluated against a clear current-state reference point.

### Design Principles and Data Governance (SCOR Enable)

Before computing KPIs, two helper functions are introduced:

- `safe_sum()` ensures total-volume metrics (e.g., revenue, profit, value-at-risk) remain reliable even if some fields arrive as strings or contain unexpected values. Non-numeric entries are coerced to missing and treated as zero for aggregation.

- `safe_mean()` provides stable averages for rate and margin metrics by coercing values to numeric prior to averaging.

These safeguards represent SCOR Enable best practice. The functions ensure that the resulting KPIs are robust and reproducible - a core requirement for any decision-oriented supply chain KPI framework.

### SCOR KPI Framework: What Each Metric Represents

#### 1) PLAN - Demand Analytics   

- `plan_demand_volatility_cv`: Coefficient of Variation (CV) of weekly net sales.   
  
CV is a compact measure of demand volatility relative to the typical demand level. Higher values indicate less predictable demand, which increases planning complexity and impacts forecasting accuracy, inventory strategy, and capacity decisions.
  
This KPI anchors the SCOR Plan narrative:   

- **High Volatility:** Complicates forecasting accuracy and necessitates higher safety stock buffers to protect service levels (inventory availability).
- **Low Volatility:** Enhances demand predictability, enabling the aggressive optimization of capacity and inventory flow - the frequency and timing of inventory restocking. This stability allows for leaner operations and reduced safety stock requirements.

*Note:* Weekly aggregation is deliberate. It aligns with common planning cadences and avoids overreacting to day-to-day noise.

#### 2) SOURCE/MAKE - Internal Efficiency (Proxy)

- `source_avg_processing_days`: Average time between order placement and shipping date.   

This is a proxy for internal business execution speed - how quickly the organization can convert demand into outbound fulfillment. While it doesn’t isolate procurement vs production vs picking/packing, it does measure a highly meaningful operational outcome: order-to-ship responsiveness.

Processing time (Internal Cycle Time) is often one of the most actionable levers in supply chain improvement:

- Longer processing time increases total lead time and the likelihoood of a late delivery.
- Improvements here can directly reduce WIP (Work-in-Process) - the volume of floating orders that have been received and paid for but have not yet exited the warehouse - thereby clearing backlogs and shortening customer wait times.

This metric supports SCOR Source/Make by quantifying internal efficiency in the absence of detailed factory/procurement timestamps.

#### 3) DELIVER - Service Level and Reliability

- **Late Delivery Rate (`deliver_late_delivery_rate`):** Leverages the `late_delivery_risk` flag to quantify the frequency of service-level breaches - where late orders are viewed as contractual or customer-promise failures. This is the primary metric for supply chain reliability, identifying the percentage of orders that failed the perfect order requirement for on-time delivery.

- **On-time-rate (`deliver_on_time_rate`):** Defined as the inverse of the late delivery rate (1 - `late_delivery_rate`), providing an intuitive service-level metric that aligns with decision-making reporting standards.

- **Median Shipping Days (`deliver_p50_shipping_days`):** Represents the Standard Fulfillment Velocity - the baseline lead time experienced by the majority of customers. This metric defines the normal performance and the standard promise to the customer, serving as the primary benchmark for assessing the competitiveness of the Deliver process against market standards.


- **90th Percentile Shipping Days (`deliver_p90_shipping_days`):** Represents the Risk-Adjusted Lead Time - the delivery window the business can commit to with 90% confidence. By quantifying the worst-case scenario, this metric captures long-tail variability (orders delayed by logistics bottlenecks). It serves as a critical stress test for establishing safety stock levels (ensuring inventory availability) and maintaining realistic customer service commitments.


Both the p50 and p90 matter because averages can hide variability. The p50/p90 pair gives a clear view of:

- **Typical performance vs. worst-case experience:** Defines the gap between what is promised and what is actually delivered.
- **Systemic vs. tail-driven issues:** A high p50 suggests the entire Deliver process is slow (systemic), while a high p90 with a low p50 suggests the process is generally good but prone to black swan disruptions (tail-driven).

This distinction determines whether the organization should redesign the entire workflow or simply increase safety stock to buffer against the tail.

Together these KPIs align directly with SCOR Deliver: reliability and responsiveness.

#### 4) RETURN - Reverse Logistics Impact (Proxy)

- **Return rate (`return_rate`):** Calculated using non-standard `order_status` labels (e.g., `REBATE`, `CANCELLED`, `RETURN`, `SUSPECTED_FRAUD`). This metric serves as a high-level indicator of Reverse Logistics volume. High return rates often signal issues in the Source or Make phases (e.g., poor product quality) or the Deliver phase (e.g., shipping errors), directly impacting the **Total Cost to Serve** - the sum of all costs incurred to get a product from the initial Plan phase into the customer's hands and through any potential Return.

- **Return margin impact (`return_margin_impact`):** Quantifies the **Direct Profit Leakage** by summing the `order_profit_per_order` for all non-standard fulfillment outcomes (returns, cancellations, and rebates). This metric is critical because it identifies the Sunken Cost - where the Total Cost to Serve (labor, shipping, and processing) was already spent, but the revenue was never realized or was subsequently clawed back.

In many operational exports, returns lack a dedicated `returned_flag` field. Leveraging status codes provides a defensible proxy for Reverse Logistics events, allowing for a realistic assessment of commercial losses and their impact on the Total Cost to Serve. This identifies the specific points where operational failures erode net profitability.

In this context, returns and cancellations are viewed as strategic challenges rather than mere system logs; they directly drive inventory distortion, compromise customer trust, and serve as the primary challenge to net profitability. This approach aligns with the SCOR Return pillar by quantifying both the frequency of failure (Rate) and its subsequent financial erosion (Margin Impact).

#### 5) MEASURE - Financial Performance and Leakage

- **Total Net Sales (`measure_total_net_sales`):** Calculated by summing `sales_per_customer`, this metric represents Actual Price Realisation (revenue after discounts). It serves as the authoritative top-line baseline for the SCOR Measure phase, allowing for the accurate assessment of financial performance across subsequent dashboards and scorecards - strategic performance management tools used to track the supply chain's progress against high-level business goals.

- **Total Profit (`measure_total_profit`):** Calculated by summing `order_profit_per_order`, this metric captures the bottom-line contribution at the transactional level. It serves as the definitive measure of operational success within the SCOR Measure phase, reflecting the net financial outcome after all operational costs and price concessions are deducted from revenue.

- **Average Profit Margin (`measure_avg_profit_margin`):** Calculated as the mean of `profit_margin`, this metric normalises profitability to enable apples-to-apples comparisons across segments and time. It serves as a key indicator of operational efficiency within the SCOR Measure phase, identifying which areas of the business are generating the highest relative return on the Total Cost to Serve.

- **Late Delivery Value at Risk (`measure_late_delivery_value_at_risk`):** Calculated by summing the sales of all orders flagged as late, this metric quantifies the revenue exposure tied to service failures. It serves as a highly actionable Profit Leakage indicator within the SCOR Measure phase, identifying the specific financial weight of unreliability across different customer segments and markets.

    This metric provides a critical bridge between SCOR Deliver and SCOR Measure by translating operational failures into financial exposure. Late deliveries are recognized as a primary driver of commercial erosion, including immediate costs like refunds or credits, and long-term impacts such as customer churn, lost repeat purchases, and significant reputational damage.

### Output and Downstream Use

The structured KPI dictionary serves as the operational anchor for the project. By exporting these metrics, a standardized benchmark is established that ensures downstream models (Forecasting/Risk) and dashboards (Profitability/Leakage) are evaluated against a data-driven current-state baseline.

In [3]:
import pandas as pd
import numpy as np
import json

# SCOR Enable: Data quality safeguards for KPI computation
def safe_sum(series):
    """Ensures total volume metrics are accurate by handling non-numeric data."""
    return float(pd.to_numeric(series, errors = "coerce").fillna(0).sum())

def safe_mean(series):
    """Provides a reliable average for margin and rate analysis."""
    return float(pd.to_numeric(series, errors = "coerce").mean())

# SCOR Operational Baseline KPIs 
kpis = {}

#  PLAN: Demand Analytics
if "sales_per_customer" in df.columns:
    weekly_sales = df.groupby(df['order_date'].dt.to_period('W'))['sales_per_customer'].sum()
    kpis["plan_demand_volatility_cv"] = float(weekly_sales.std() / weekly_sales.mean())

#  SOURCE/MAKE: Internal Efficiency
if "processing_days" in df.columns:
    kpis["source_avg_processing_days"] = safe_mean(df["processing_days"])

#  DELIVER: Service Level and Reliability
if "late_delivery_risk" in df.columns:
    kpis["deliver_late_delivery_rate"] = float(df["late_delivery_risk"].mean())
    kpis["deliver_on_time_rate"] = 1.0 - kpis["deliver_late_delivery_rate"]

if "days_for_shipping_real" in df.columns:
    # Responsiveness: Median (p50) and Risk-Adjusted (p90) lead times
    kpis["deliver_p50_shipping_days"] = float(df["days_for_shipping_real"].median())
    kpis["deliver_p90_shipping_days"] = float(df["days_for_shipping_real"].quantile(0.9))

# RETURN: Reverse Logistics Impact - proxied via non-standard order statuses
if "order_status" in df.columns:
    return_mask = df['order_status'].isin(['REBATE', 'CANCELLED', 'RETURN', 'SUSPECTED_FRAUD'])
    kpis["return_rate"] = float(return_mask.mean())
    kpis["return_margin_impact"] = safe_sum(df.loc[return_mask, 'order_profit_per_order'])

#  MEASURE: Financial Performance and Leakage
if "sales_per_customer" in df.columns:
    kpis["measure_total_net_sales"] = safe_sum(df["sales_per_customer"])
    
    # Revenue tied to late deliveries - Value at Risk
    late_mask = df['late_delivery_risk'] == 1
    kpis["measure_late_delivery_value_at_risk"] = safe_sum(df.loc[late_mask, "sales_per_customer"])

if "order_profit_per_order" in df.columns:
    kpis["measure_total_profit"] = safe_sum(df["order_profit_per_order"])

if "profit_margin" in df.columns:
    kpis["measure_avg_profit_margin"] = safe_mean(df["profit_margin"])

# Output baseline as structured JSON
print("SCOR Operational Baseline:")
print(json.dumps(kpis, indent = 4))

SCOR Operational Baseline:
{
    "plan_demand_volatility_cv": 0.21878840633986216,
    "source_avg_processing_days": 3.4739144858321387,
    "deliver_late_delivery_rate": 0.5488379662374893,
    "deliver_on_time_rate": 0.45116203376251074,
    "deliver_p50_shipping_days": 3.0,
    "deliver_p90_shipping_days": 6.0,
    "return_rate": 0.022364377233777533,
    "return_margin_impact": 73108.389991526,
    "measure_total_net_sales": 29052383.16661395,
    "measure_late_delivery_value_at_risk": 15896632.512272123,
    "measure_total_profit": 3479418.243254454,
    "measure_avg_profit_margin": 0.1201800019408187
}


### SCOR Operational Baseline - Phase-by-Phase Interpretation

The computed KPIs represent a current-state operational baseline, quantifying how the supply chain performs across planning stability, execution efficiency, delivery reliability, reverse logistics exposure, and financial outcomes. Interpreted together, they reveal a supply chain that is commercially viable but operationally fragile, with reliability emerging as the dominant risk driver.

#### PLAN - Demand Stability and Forecastability

Demand Volatility (CV) → 0.219

In the SCOR Plan process, a CV under 0.25 is generally considered stable to moderate. A CV of 0.22 indicates that while demand is structurally stable at a weekly cadence, standard rule-based systems are insufficient to manage the 22% variance. This predictability gap justifies the transition to advanced predictive modeling to optimize safety stock levels and replenishment cycles.

Strategic Implications:

- **Model Viability:** The stable CV suggests that forecasting models will achieve high accuracy, provided they incorporate appropriate seasonality and trend handling.
- **Resource Optimization:** While safety stock and capacity buffers are necessary to mitigate the 22% variance, they do not need to be excessive, allowing for leaner asset management.
- **Continuous Improvement:** Even minor gains in demand sensing and forecasting accuracy will yield meaningful downstream benefits by reducing **Inventory Days of Supply** and improving the **Cash-to-Cash Cycle Time**.

##### SOURCE/MAKE - Internal Processing Efficiency

Average Processing Time → ~3.47 days.

On average, orders reside for 3.5 days in internal processing before transitioning to the carrier. This duration serves as a holistic measure of **Order-to-Ship Responsiveness**, capturing the combined efficiency of sourcing, warehouse picking, and dispatch.

Strategic Implications:

- **Lead Time Compression:** Internal processing represents a significant portion of the total **Order Fulfillment Cycle Time**. Even marginal improvements here would create a time buffer with the potential to reduce late delivery risk.
- **Operational Control:** Unlike transit delays, this metric is largely under internal control, making it the most actionable lever for improving Service Levels.
- **Throughput Constraint:** Within the SCOR Source/Make context, this 3.5-day baseline suggests a structural bottleneck that likely compounds delivery risk during demand spikes or operational disruptions.

#### DELIVER - Service Reliability and Responsiveness

- Late Delivery Rate: ~55%
- On-Time Delivery Rate: ~45%
- Median Delivery Time (p50): 3 days
- Risk-Adjusted Delivery Time (p90): 6 days

Service reliability is the primary strategic weakness in the current-state baseline. The data reveals a reliability-velocity paradox: the supply chain is functionally fast - with a median delivery time of just 3 days - yet operationally unreliable, with 55% of orders failing to meet their scheduled commitments. The 90th percentile extending to 6 days confirms that the Standard Promise is frequently compromised by a pronounced long tail of delayed shipments.

Strategic Implications:

- **Velocity vs. Reliability:** The process is fast on average but lacks the process stability required for consistent service outcomes.
- **Systemic Variability:** The 55% failure rate proves that late deliveries are systemic process failures rather than isolated black swan events; the current scheduled window is fundamentally disconnected from actual real performance.
- **Commercial Erosion:** From a SCOR Deliver perspective, this represents a significant reliability failure that directly erodes customer trust and Lifetime Value (LTV).

*Note:* This 3-to-6 day Responsiveness Gap is the primary driver of Profit Leakage, necessitating the transition from descriptive tracking - Summarising historical data into KPIs - to predictive risk modeling.

##### RETURN - Reverse Logistics and Margin Exposure

- Return Rate: ~2.2%
- Margin Impact from Returns: ~$73k

The Return Rate is relatively low (~2.2%), while the Margin Impact of ~$73k reveals a significant Efficiency Tax on the bottom line. This suggests that non-standard fulfillment outcomes - returns, cancellations, and rebates - disproportionately erode the profitability gained through the Deliver process.

Strategic Implications:

- **Concentrated Erosion:** The financial weight of these events suggests that Reverse Logistics costs may be concentrated in high-value products or specific high-risk markets.
- **The Multiplier Effect:** Even low-frequency return events act as a **negative multiplier**, as the business loses the revenue while still incurring the Total Cost to Serve (shipping, handling, and processing).
- **Root-Cause Priority:** Shifting focus from Return Volume to Return Rationale is important; identifying whether these are driven by product quality (Source/Make) or delivery delays (Deliver) is the only way to protect the margin.

*Note:* Within the SCOR Return phase, this identifies a major opportunity to synchronize delivery reliability with reverse-logistics prevention. High-risk shipments should be monitored as Leading Indicators for future return costs.

##### MEASURE - Financial Performance and Value at Risk

- Total Net Sales: ~\$29.1M
- Total Profit: ~\$3.48M
- Average Profit Margin: ~12%.
- Late Delivery Value at Risk: ~\$15.9M

The supply chain demonstrates robust financial health at scale, with an Average Profit Margin of 12% and over $3.48M in Total Profit. However, the Late Delivery Value at Risk reveals a staggering Exposure Gap: more than 50% of the total revenue is generated through orders that fail to meet service-level expectations. This suggests that while the business is profitable, its revenue foundation is highly unstable.

Strategic Implications:

- **Vulnerability to Churn:** Profitability is currently subsidized by customers who accept late deliveries, but this creates a Service-Level Vulnerability that competitors can easily exploit.
- **Financial Exposure as a Catalyst:** The $15.9M at risk transforms late shipping from a logistics nuisance into a primary Financial Risk that threatens top-line stability and investor confidence.
- **The Growth Paradox:** Protecting this revenue does not require new customer acquisition (Marketing spend); it requires Operational Reliability (Process spend). Improving delivery could secure over 50% of current sales without a single amount in additional customer acquisition cost.
  
This SCOR Measure insight proves that reliability is not a cost center, but rather a Revenue Protection strategy. Quantifying the Value at Risk creates a definitive business case for investing in predictive interventions.

#### Cross-SCOR Synthesis

Taken together, the Operational Baseline reveals a consistent narrative of a supply chain that is scaling but fragile:

- **Manageable Volatility:** Demand is structurally stable (CV ~0.22), confirming that the current service failure is not a Planning issue, but an Execution issue.
- **Internal Latency:** The 3.5-day processing bottleneck consumes critical lead-time, leaving the Deliver process with zero logistical runway to handle transit variability.
- **Systemic Unreliability:** The 55% late delivery rate is the dominant failure mode, transforming the Deliver phase into a primary driver of commercial erosion.
- **Financial Exposure:** Profitability is robust at 12%, but it is inherently fragile, with over 50% of revenue ($15.9M) currently tied to service-level breaches.

#### Strategic Verdict:

The data confirms that **internal execution bottlenecks** and **service variability**, rather than unpredictable market demand, are the primary drivers of risk. This establishes a definitive Analytical Roadmap: the focus must shift from **Descriptive Tracking** to **Predictive Risk Modelling** to protect the $15.9M at risk and quantify the ROI of operational stability.

## Weekly Revenue Performance and Regional Market Dynamics

To enable market-level demand planning and regional performance analysis, transactional sales data is aggregated into a weekly net sales time series by market. This transformation aligns the analytical grain with the SCOR Plan process, supporting both strategic forecasting and comparative reporting across geographic markets.

Standardizing demand at a weekly cadence and explicitly labeling the metric as `weekly_net_sales` ensures the resulting Gold dataset provides a clear, unambiguous definition of demand that can be consistently reused across Power BI dashboards, forecasting models, and forecast accuracy evaluations.

In [4]:
# # Validate required columns from Silver layer 
if ("order_date" in df.columns and 
    "sales_per_customer" in df.columns and 
    "market" in df.columns):
    
    # Subset to Plan-relevant fields
    demand = df[["order_date", "sales_per_customer", "market"]].copy()
    
    # Roll up to weekly grain - SCOR Plan process standard
    demand["order_week"] = demand["order_date"].dt.to_period("W").dt.start_time
    
     # Aggregate to market-level weekly net sales
    demand_weekly_by_market_gold = (
        demand.groupby(["order_week", "market"])["sales_per_customer"]
        .sum()
        .reset_index()
        .rename(columns = {"sales_per_customer": "weekly_net_sales"})
    )

    display(demand_weekly_by_market_gold.head())
else:
    print("Required columns missing from the Silver Table.")

,order_week,market,weekly_net_sales
0,2014-12-29,LATAM,105099.331155
1,2015-01-05,LATAM,186178.672183
2,2015-01-12,LATAM,185840.851909
3,2015-01-19,LATAM,186720.131978
4,2015-01-26,LATAM,187502.662140


## Persist Gold Datasets

Two Gold-layer assets are persisted from this notebook:

**Weekly Demand by Market** - a standardized time series aggregated to the SCOR Plan process grain. This dataset provides a single, governed definition of demand for reuse across Power BI dashboards, forecasting models, and forecast-versus-actual evaluations.

**SCOR KPI Baseline** - a structured dictionary of current-state performance metrics spanning Plan, Source/Make, Deliver, Return, and Measure. This baseline serves as the benchmark against which all downstream model outputs and process improvements are evaluated.

Storing both datasets independently of downstream models ensures that reporting and predictive outputs share a common, reproducible reference point, eliminating metric divergence between analytics layers.

In [5]:
gold_demand_weekly_file_path = "gold_demand_weekly_by_market.parquet"
gold_scor_baseline_file_path = "gold_scor_baseline.parquet"

# Save Gold Demand Planning table
try:
    demand_weekly_by_market_gold.to_parquet(gold_demand_weekly_file_path, index = False)
    print(f"Gold Weekly Demand Table Saved: {gold_demand_weekly_file_path}")
    print(f"Final Grain: Weekly Demand | Records: {len(demand_weekly_by_market_gold)}")
except Exception as e:
    print(f"Failed to Save Gold Weekly Demand Table: {e}")

# Save SCOR KPI baseline
try:
    pd.DataFrame([kpis]).to_parquet(gold_scor_baseline_file_path, index = False)
    print(f"Gold SCOR KPIs Saved: {gold_scor_baseline_file_path}")
    print(f"Final Grain: SCOR KPIs | Records: {len(kpis)}")
except Exception as e:
    print(f"Failed to Save Gold SCOR KPIs: {e}")

Gold Weekly Demand Table Saved: gold_demand_weekly_by_market.parquet
Final Grain: Weekly Demand | Records: 234
Gold SCOR KPIs Saved: gold_scor_baseline.parquet
Final Grain: SCOR KPIs | Records: 12
